In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install biopython
!pip install obonet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.5 MB/s eta 0:00:00


In [3]:
import warnings

warnings.filterwarnings("ignore")

import torch
import numpy as np
from transformers import BertModel, BertTokenizer
from tqdm import tqdm
from Bio import SeqIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False )
model = BertModel.from_pretrained("Rostlab/prot_bert").to(device)
model.eval()

class Config:
    IA_path  = "/kaggle/input/cafa-6-protein-function-prediction/IA.tsv"
    train_terms = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
    train_taxonomy = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv"
    train_sequences = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
    go_basic = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
    test_sequences = "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta"

config = Config()


2025-12-24 11:16:14.592541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766574974.794380      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766574974.851952      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766574975.325760      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766574975.325800      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766574975.325803      24 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

In [4]:
def load_fasta_as_dataframe(fasta_path: str, train_data:bool = True):
    ids,sequences = [],[]

    for record in SeqIO.parse(fasta_path, "fasta"):
        if train_data:
            uniprot_id = record.id.split("|")[1]
        else:
            uniprot_id = record.id
            
        ids.append(uniprot_id)
        sequences.append(str(record.seq))

    return pd.DataFrame({
        "EntryID": ids,
        "sequences": sequences
    })


# ===== Usage =====
train_sequences_df = load_fasta_as_dataframe(config.train_sequences, train_data=True)
test_sequences_df = load_fasta_as_dataframe(config.test_sequences, train_data=False)
train_sequences_df


,EntryID,sequences
0,A0A0C5B5G6,MRWQEMGYIFYPRKLR
1,A0JNW5,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...
2,A0JP26,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...
3,A0PK11,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...
4,A1A4S6,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...
...,...,...
82399,Q9UTM1,MSKLKAQSALQKLIESQKNPNANEDGYFRRKRLAKKERPFEPKKLV...
82400,Q9Y7I1,MSSNSNTDHSTGDNRSKSEKQTDLRNALRETESHGMPPLRGPAGFP...
82401,Q9Y7P7,MRSNNSSLVHCCWVSPPSLTRLPAFPSPRILSPCYCYNKRIRPFRG...
82402,Q9Y7Q3,MHSSRRKYNDMWTARLLIRSDQKEEKYPSFKKNAGKAINAHLIPKL...


In [5]:
def generate_protBert_embeddings(sequences_df, 
                               seq_col="sequences",
                               id_col="EntryID",
                               max_len=1000,
                               batch_size=4):
  

    EntryIDs = sequences_df[id_col].tolist()
    sequences = sequences_df[seq_col].tolist()

    # space-separated amino acids for ProtBert
    sequences = [" ".join(list(str(seq))) for seq in sequences]

    ids, embeds = [], []

    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i + batch_size]
        batch_ids  = EntryIDs[i:i + batch_size]

        tokens = tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        )

        input_ids = tokens["input_ids"].to(device)
        attention_mask = tokens["attention_mask"].to(device)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Mean pooling over sequence length
        batch_embs = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

        ids.extend(batch_ids)
        embeds.extend(batch_embs)

    return np.array(ids), np.array(embeds)


In [6]:
torch.cuda.empty_cache()


# test_ids, test_embeds = generate_protBert_embeddings(
#     test_sequences_df,
#     seq_col="sequences",
#     id_col="EntryID",
#     max_len=1000,
#     batch_size=128
# )


train_ids, train_embeds = generate_protBert_embeddings(
    train_sequences_df,
    seq_col="sequences",
    id_col="EntryID",
    max_len=1000,
    batch_size=128
)

print(train_embeds.shape)   # (N, 1024)

100%|██████████| 644/644 [3:10:47<00:00, 17.78s/it]

(82404, 1024)


In [7]:
np.save("train_ids.npy", train_ids)
np.save("train_embeds.npy", train_embeds)